In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, ParameterSampler, cross_val_score, ParameterGrid
import numpy as np
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
import joblib
import time

# Consts

In [ ]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"]

# Initiate pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])

# Init wandb

In [ ]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

# Randomized search

In [ ]:
ngram_ranges_word = [(1, 2), (1, 3), (2, 3)]
ngram_ranges_char = [(2, 4), (3, 5), (4, 6), (2, 5)]

rnd_params = {
    "features__tfidf_word__use_idf": [True, False],
    "features__tfidf_word__sublinear_tf": [True, False],
    "features__tfidf_word__norm": ["l1", "l2"],
    "features__tfidf_word__max_df": uniform(0.55, 0.45),
    "features__tfidf_word__min_df": uniform(0.0, 0.45),
    "features__tfidf_word__max_features": randint(5000, 60001),
    "features__tfidf_word__ngram_range": ngram_ranges_word,

    "features__tfidf_char__use_idf": [True, False],
    "features__tfidf_char__sublinear_tf": [True, False],
    "features__tfidf_char__norm": ["l1", "l2"],
    "features__tfidf_char__max_df": uniform(0.6, 0.3),
    "features__tfidf_char__min_df": uniform(0.001, 0.049),
    "features__tfidf_char__max_features": randint(5000, 60001),
    "features__tfidf_char__ngram_range": ngram_ranges_char,

    "svd__n_components": randint(10, 600),

    "clf__C": loguniform(1e-2, 1e2),
}

In [ ]:
rnd_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search",group="random_search")

seed = int(os.getenv("RANDOM_SEED", 880055535))
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=seed
)
sampler = ParameterSampler(
    rnd_params,
    n_iter=1000,
    random_state=seed
)
best_score = -1
best_rnd = dict()
for i, params in enumerate(sampler):

    pipeline.set_params(**params)

    time_start = time.time()

    scores = cross_val_score(
        pipeline,
        x_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=4
    )

    mean_score = scores.mean()
    time_elapsed = time.time() - time_start
    wandb.log({
        "iteration": i,
        "time_elapsed": time_elapsed,
        "score": mean_score,
        **params
    })

    print({"iteration": i, "time_elapsed": time_elapsed, "score": mean_score, **params})

    if mean_score > best_score:
        best_score = mean_score
        best_rnd = params

wandb.log({
    "best_score": best_score,
    "best_params": best_rnd
})

rnd_run.finish()

# Grid search

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    return uniq

grid_params = {
    "features__tfidf_word__use_idf": [best_rnd["features__tfidf_word__use_idf"]],
    "features__tfidf_word__sublinear_tf": [best_rnd["features__tfidf_word__sublinear_tf"]],
    "features__tfidf_word__norm": [best_rnd["features__tfidf_word__norm"]],
    "features__tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "features__tfidf_word__max_df": make_range(best_rnd["features__tfidf_word__max_df"]),
    "features__tfidf_word__min_df": make_range(best_rnd["features__tfidf_word__min_df"]),
    "features__tfidf_word__max_features": make_range(best_rnd["features__tfidf_word__max_features"], cast_int=True),

    "features__tfidf_char__use_idf": [best_rnd["features__tfidf_char__use_idf"]],
    "features__tfidf_char__sublinear_tf": [best_rnd["features__tfidf_char__sublinear_tf"]],
    "features__tfidf_char__norm": [best_rnd["features__tfidf_char__norm"]],
    "features__tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "features__tfidf_char__max_df": make_range(best_rnd["features__tfidf_char__max_df"]),
    "features__tfidf_char__min_df": make_range(best_rnd["features__tfidf_char__min_df"]),
    "features__tfidf_char__max_features": make_range(best_rnd["features__tfidf_char__max_features"], cast_int=True),

    "svd__n_components": [best_rnd["svd__n_components"]],

    "clf__C": make_range(best_rnd["clf__C"]),
}

In [ ]:
grid_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search",group="grid_search")

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
best_score = -1
best_params = dict()
best_model = None

grid = ParameterGrid(grid_params)
for i, params in enumerate(grid):

    pipeline.set_params(**params)

    scores = cross_val_score(
        pipeline,
        x_train,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=4
    )

    mean_score = scores.mean()

    wandb.log({
        "iteration": i,
        "score": mean_score,
        **params
    })

    print({"iteration": i, "score": mean_score, **params})

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

wandb.log({
    "best_score": best_score,
    "best_params": best_params
})

model_path = os.path.join(MODEL_DIR, "grid_model.joblib")

best_model = pipeline.set_params(**best_params).fit(x_train, y_train)
joblib.dump(best_model, model_path)

if os.path.exists(model_path):
    print(f"Model saved to {model_path}")

    artifact = wandb.Artifact("best_model", type="model")
    artifact.add_file(model_path)

    wandb.log_artifact(artifact)

grid_run.finish()